# Core Techniques — Prompt Engineering

### The #1 Skill Employers Want From GenAI Engineers

---

This notebook is the **hands-on companion** to **Section 2: Prompt Engineering** in the course materials — the same topics as **Getting Started** (*Talking to an LLM*, `pe-api` on the page) and **Core Techniques** (Chapters **1–4**: Zero-Shot, Few-Shot, Role & System Prompts, Chain-of-Thought). 

**The shift:** Once you understand how the engine works (see `1.How_The_Model_Works_Inference.ipynb` in this folder), you learn to drive it.

**Why this matters for your career:** Every GenAI job posting mentions prompt engineering. It's the skill that turns a generic LLM into a useful product. Two engineers using the same model can get dramatically different results based purely on how they write their prompts.

**In this notebook:**

0. **Talking to an LLM (Getting Started)** — install, client, `chat.completions.create`, response object, token usage  
1. **Zero-Shot Prompting** — just ask, no examples  
2. **Few-Shot Prompting** — teach by showing examples  
3. **Role Prompting & System Prompts** — set the model's persona  
4. **Chain-of-Thought** — make the model think step by step  

**How we'll learn:** Every technique is demonstrated with a real problem. For each one, we'll first see the **naive prompt** (what most people would write), then the **engineered prompt** (using the technique), and compare the results side by side.

**Model:** We'll use `gpt-4o-mini` throughout — it's the cheapest option ($0.15/1M input tokens) and good enough to demonstrate all techniques clearly. In production, the same techniques work even better with larger models.

---


---
## Part 0: Talking to an LLM (Getting Started)

On the **Prompt Engineering** page, this matches **Getting Started → Talking to an LLM** (`#pe-api`).

Before the prompting techniques, you need the **mechanics** of calling an LLM from Python. We use the **OpenAI API** (same stack as ChatGPT). Anthropic, Google, and others follow nearly the same pattern: a client, a chat call, and a structured response object.

**Hands-on:** Run the cells in order. You need an API key from [platform.openai.com/api-keys](https://platform.openai.com/api-keys).


### Step 1: Install the OpenAI library

One package handles HTTP, auth, retries, and parsing.

In [ ]:
%pip install -q openai


### Step 2: Set up the client

**Recommended:** set the environment variable `OPENAI_API_KEY` before starting Jupyter (or use Colab secrets / your host's env settings).

**Notebook-only (not for shared repos):** you can set it in the next cell temporarily — **never commit real keys to git.**


In [10]:
import os
from openai import OpenAI

# If OPENAI_API_KEY is not already set in your environment, set it here ONCE for this session:
# os.environ["OPENAI_API_KEY"] = "sk-..."  # set only for local demo; never commit keys

if not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError(
        "Set OPENAI_API_KEY first. Example: export OPENAI_API_KEY='sk-...' in the terminal, "
        "or os.environ['OPENAI_API_KEY'] = 'sk-...' in this notebook (do not commit keys)."
    )

client = OpenAI()  # reads API key from the environment
print("Client ready.")


Client ready.


### Step 3: Your first `chat.completions.create` call

The **minimum** is `model` + `messages`. Each message has a `role` (`system`, `user`, or `assistant`) and `content` (the text).


In [11]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": "What is Hadoop? Answer in one short sentence."}
    ],
)

print(response.choices[0].message.content)


Hadoop is an open-source framework that allows for the distributed processing and storage of large data sets across clusters of computers using simple programming models.


### Full call with common parameters

In practice you often pass **`temperature`** (creativity), **`max_tokens`** (length cap), **`top_p`**, and a **`system`** message for persistent instructions.


In [12]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a helpful data engineer. Be concise."},
        {"role": "user", "content": "What is Hadoop?"},
    ],
    temperature=0.7,
    max_tokens=500,
    top_p=0.9,
)

print(response.choices[0].message.content)


Hadoop is an open-source framework for distributed storage and processing of large datasets using clusters of commodity hardware. It consists of the Hadoop Distributed File System (HDFS) for storing data and the MapReduce programming model for processing it. Hadoop enables scalable, fault-tolerant data management and analytics, making it suitable for big data applications.


### Message roles (conversation format)

| Role | Purpose |
|------|--------|
| **`system`** | Developer instructions: persona, rules. Highest priority; applies to the whole chat. |
| **`user`** | The end user's prompt or task. |
| **`assistant`** | Prior model turns — **you** paste these back in for multi-turn context. |

The model was trained on `(system, user, assistant)` dialogues (SFT), which is why the API looks like this.

**Important:** The model has **no memory between API calls**. Each request is stateless. For a chat UI you must send the **full** `messages` list every time.


### Multi-turn example

The model only "remembers" because we resend earlier turns in `messages`:

In [13]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a Python tutor."},
        {"role": "user", "content": "What is a list?"},
        {"role": "assistant", "content": "A list is an ordered, mutable collection in Python."},
        {"role": "user", "content": "How do I add items to one?"},
    ],
)
print(response.choices[0].message.content)


You can add items to a list in Python using several methods. Here are the most common ones:

1. **Using the `append()` method**: This adds a single item to the end of the list.

   ```python
   my_list = [1, 2, 3]
   my_list.append(4)
   print(my_list)  # Output: [1, 2, 3, 4]
   ```

2. **Using the `extend()` method**: This adds multiple items (from another iterable) to the end of the list.

   ```python
   my_list = [1, 2, 3]
   my_list.extend([4, 5])
   print(my_list)  # Output: [1, 2, 3, 4, 5]
   ```

3. **Using the `insert()` method**: This adds an item at a specified index in the list.

   ```python
   my_list = [1, 2, 3]
   my_list.insert(1, 1.5)  # Insert 1.5 at index 1
   print(my_list)  # Output: [1, 1.5, 2, 3]
   ```

4. **Using the `+` operator**: You can concatenate two lists to create a new list.

   ```python
   my_list = [1, 2, 3]
   new_list = my_list + [4, 5]
   print(new_list)  # Output: [1, 2, 3, 4, 5]
   ```

5. **Using the `*` operator**: If you want to add the sam

### Step 4: What's inside the response?

Besides `.choices[0].message.content`, you get:

- **`finish_reason`**: `"stop"` (normal), `"length"` (hit `max_tokens` — output may be cut off), `"content_filter"` (blocked).
- **`usage`**: `prompt_tokens`, `completion_tokens`, `total_tokens` — what you're billed on.
- **`model`**: exact model version that ran (useful for logs).
- **`id`**: unique id for this completion (debugging / support).

Run the next cells to inspect a real object.


In [14]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "What is Hadoop? One sentence."}],
    max_tokens=120,
)

choice = response.choices[0]
msg = choice.message

print("message.role:", msg.role)
print("message.content:", msg.content)
print("finish_reason:", choice.finish_reason)
print("model:", response.model)
print("id:", response.id)
print("usage:", response.usage)


message.role: assistant
message.content: Hadoop is an open-source framework that enables the distributed processing of large data sets across clusters of computers using simple programming models.
finish_reason: stop
model: gpt-4o-mini-2024-07-18
id: chatcmpl-DOggrKcjpfDfeMUnhlCuWP7IAHQmY
usage: CompletionUsage(completion_tokens=25, prompt_tokens=14, total_tokens=39, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0))


### Production-style handling

Always check `finish_reason` for truncation, and log token usage for cost tracking.

In [15]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "What is Hadoop?"}],
    max_tokens=500,
)

answer = response.choices[0].message.content
fr = response.choices[0].finish_reason

if fr == "length":
    print("⚠️ Response may be truncated — increase max_tokens.")
elif fr == "content_filter":
    print("⚠️ Response blocked by safety filter.")

print(answer)
print(
    f"Tokens: {response.usage.prompt_tokens} in + "
    f"{response.usage.completion_tokens} out = {response.usage.total_tokens} total"
)
print("Call ID:", response.id)


⚠️ Response may be truncated — increase max_tokens.
Hadoop is an open-source framework that is used for storing and processing large datasets across clusters of computers using simple programming models. It is designed to scale up from a single server to thousands of machines, each offering local computation and storage. Here are some of its key components and features:

1. **Hadoop Distributed File System (HDFS)**: This is the storage component of Hadoop. HDFS is designed to store large files across multiple machines, providing fault tolerance and high throughput access to application data. It divides the files into blocks (default size: 128MB or 256MB) and distributes them across the cluster.

2. **MapReduce**: This is the processing model used in Hadoop. MapReduce allows for the parallel processing of large datasets by dividing the task into smaller sub-tasks (map tasks) that can be processed in parallel, and then combining the results (reduce tasks). This model handles failures and

### Helper we'll use for the rest of this notebook

To avoid repeating boilerplate, we define **`ask()`**: it sends one user message (optional system prompt), prints the reply and token line, and returns the text.

*(Pricing shown is a rough estimate for `gpt-4o-mini` input-heavy mixes; check current pricing on OpenAI's site.)*


In [16]:
def ask(prompt, system_prompt=None, temperature=0.3, model="gpt-4o-mini", max_tokens=1000):
    """Send a prompt to the model, print the response and token summary, return the text."""
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": prompt})

    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens,
    )

    result = (response.choices[0].message.content or "").strip()
    usage = response.usage
    total = usage.total_tokens if usage else 0

    print(result)
    print(f"\n{'─' * 50}")
    # Rough $ estimate: mini ~ $0.15/1M input, ~$0.60/1M output — adjust if pricing changes
    est = 0.0
    if usage:
        est = usage.prompt_tokens * 0.15e-6 + usage.completion_tokens * 0.60e-6
    print(f"Tokens: {total} (in {usage.prompt_tokens}, out {usage.completion_tokens}) | Est. ~${est:.6f}")

    return result


print("Setup complete. Ready for Chapter 1: Zero-Shot.")


Setup complete. Ready for Chapter 1: Zero-Shot.


---

# Chapter 1: Zero-Shot Prompting

---

*Course page: Chapter 1 — Zero-Shot (`#pe1`)*



Let's see it in action with a real task — **classifying customer support emails.**

In [3]:
# ============================================================
# ZERO-SHOT: The naive approach — just ask
# ============================================================

email = """Hi, I purchased the Pro plan recently but I'm still seeing the free tier 
limits on my account. I've been charged $49.99 on my credit card but the dashboard 
still shows 'Free Plan'. I've tried logging out and back in but nothing changed. 
This is really frustrating — I need the Pro features for a client demo tomorrow. 
Can someone please fix this ASAP?"""

print("NAIVE ZERO-SHOT PROMPT:")
print("=" * 50)
result = ask(f"Classify this support email: {email}")


NAIVE ZERO-SHOT PROMPT:
This support email can be classified as a **Billing Issue** and **Account Upgrade Problem**. The customer is experiencing a problem with their account not reflecting the Pro plan they purchased, despite being charged for it. They are also expressing urgency due to an upcoming client demo.

──────────────────────────────────────────────────
Tokens used: 146 | Est. cost: $0.000022


### What's wrong with the naive approach?

Run the cell above and look at the output. The model probably gave you a classification, but ask yourself:
- Did it use the categories YOUR team uses? Or did it make up its own?
- Did it give just the category, or a long explanation you'd need to parse?
- Would the output be consistent if you sent 1000 emails?

Let's fix this with a **better zero-shot prompt** — same technique (no examples), but much clearer instructions.

In [4]:
# ============================================================
# ZERO-SHOT: The engineered approach — clear and specific
# ============================================================

print("ENGINEERED ZERO-SHOT PROMPT:")
print("=" * 50)

result = ask(f"""Classify the following customer support email into exactly ONE of these categories:
- BILLING
- ACCOUNT_ACCESS
- BUG_REPORT
- FEATURE_REQUEST
- CANCELLATION
- OTHER

Also assign a priority: URGENT, HIGH, MEDIUM, or LOW.

Respond in this exact format:
Category: <category>
Priority: <priority>

Email: {email}""")


ENGINEERED ZERO-SHOT PROMPT:
Category: BILLING  
Priority: URGENT

──────────────────────────────────────────────────
Tokens used: 170 | Est. cost: $0.000025


### What made the second prompt better?

Three things changed, and all three matter:

1. **Defined the categories** — Instead of letting the model make up its own categories, we told it exactly what categories to use. Now the output is predictable and consistent.

2. **Constrained the output format** — "Respond in this exact format" with a template. This is crucial for production systems where you need to parse the output programmatically.

3. **Added a second dimension** — Priority alongside category. A single prompt can extract multiple pieces of information when the instructions are clear.

Let's test this on a few more emails to see if it stays consistent:

In [5]:
# ============================================================
# ZERO-SHOT: Testing consistency across multiple inputs
# ============================================================

emails = [
    "I'd love to see a dark mode option in the mobile app. Not urgent, just a suggestion!",
    "URGENT: I cannot log into my account. I've tried resetting my password 3 times. I have a board presentation in 2 hours and all my files are locked inside.",
    "Hey, I want to cancel my subscription. The product is fine but we're cutting costs this quarter. Can you process this before the next billing cycle?",
    "The export to PDF feature keeps crashing when the report has more than 50 pages. I've tried Chrome and Firefox — same issue on both.",
    "I was charged twice for my January subscription — $49.99 on Jan 1st and again on Jan 3rd. Please refund the duplicate charge.",
]

prompt_template = """Classify the following customer support email into exactly ONE of these categories:
- BILLING
- ACCOUNT_ACCESS
- BUG_REPORT
- FEATURE_REQUEST
- CANCELLATION
- OTHER

Also assign a priority: URGENT, HIGH, MEDIUM, or LOW.

Respond in this exact format:
Category: <category>
Priority: <priority>

Email: {email}"""

print("BATCH CLASSIFICATION TEST:")
print("=" * 70)
for i, e in enumerate(emails, 1):
    result = ask(prompt_template.format(email=e))
    print(f"  Email {i}: {e[:65]}...")
    print(f"  → {result}")
    print()


BATCH CLASSIFICATION TEST:
Category: FEATURE_REQUEST  
Priority: LOW

──────────────────────────────────────────────────
Tokens used: 109 | Est. cost: $0.000016
  Email 1: I'd love to see a dark mode option in the mobile app. Not urgent,...
  → Category: FEATURE_REQUEST  
Priority: LOW

Category: ACCOUNT_ACCESS  
Priority: URGENT

──────────────────────────────────────────────────
Tokens used: 128 | Est. cost: $0.000019
  Email 2: URGENT: I cannot log into my account. I've tried resetting my pas...
  → Category: ACCOUNT_ACCESS  
Priority: URGENT

Category: CANCELLATION  
Priority: URGENT

──────────────────────────────────────────────────
Tokens used: 123 | Est. cost: $0.000018
  Email 3: Hey, I want to cancel my subscription. The product is fine but we...
  → Category: CANCELLATION  
Priority: URGENT

Category: BUG_REPORT  
Priority: HIGH

──────────────────────────────────────────────────
Tokens used: 117 | Est. cost: $0.000018
  Email 4: The export to PDF feature keeps crashing when

### Key Takeaway — Zero-Shot

Zero-shot works well when:
- The task is straightforward
- You clearly define the expected output format
- The categories or options are explicit

Zero-shot struggles when:
- The task has subtle distinctions the model might miss
- You need a very specific output style that's hard to describe in words
- The model keeps misclassifying edge cases

When zero-shot isn't enough → **few-shot prompting.**

---

# Chapter 2: Few-Shot Prompting

---

*Course page: Chapter 2 — Few-Shot (`#pe2`)*


Let's tackle a harder problem — **extracting structured data from unstructured text.** This is something you'll do constantly as a GenAI Engineer.

In [6]:
# ============================================================
# ZERO-SHOT attempt at data extraction (for comparison)
# ============================================================

job_posting = """We're looking for a Senior Backend Engineer to join our Platform team 
in Austin, TX (hybrid — 3 days in office). You'll build and scale our payment 
processing infrastructure handling 2M+ transactions daily. Requirements: 5+ years 
with Python or Go, experience with PostgreSQL and Redis, familiarity with AWS (ECS, 
Lambda, SQS). Compensation: $165K-$195K base + equity. Reports to VP of Engineering. 
Visa sponsorship available."""

print("ZERO-SHOT DATA EXTRACTION:")
print("=" * 50)

result = ask(f"""Extract the key information from this job posting and return it as structured data:

{job_posting}""")


ZERO-SHOT DATA EXTRACTION:
```json
{
  "job_title": "Senior Backend Engineer",
  "team": "Platform",
  "location": "Austin, TX",
  "work_mode": "hybrid (3 days in office)",
  "responsibilities": [
    "Build and scale payment processing infrastructure",
    "Handle 2M+ transactions daily"
  ],
  "requirements": {
    "experience": {
      "years": 5,
      "languages": ["Python", "Go"],
      "databases": ["PostgreSQL", "Redis"],
      "cloud_services": ["AWS (ECS, Lambda, SQS)"]
    }
  },
  "compensation": {
    "base_salary": {
      "min": 165000,
      "max": 195000
    },
    "equity": true
  },
  "reporting_to": "VP of Engineering",
  "visa_sponsorship": true
}
```

──────────────────────────────────────────────────
Tokens used: 314 | Est. cost: $0.000047


The zero-shot attempt probably returned something, but:
- Was the format exactly what you wanted?
- Did it use the field names you'd use in your database?
- Would it produce the exact same structure for a different job posting?

Now let's use few-shot to **show** the model exactly what we want:

In [7]:
# ============================================================
# FEW-SHOT data extraction — teach by example
# ============================================================

print("FEW-SHOT DATA EXTRACTION (3 examples):")
print("=" * 50)

result = ask(f"""Extract structured information from job postings. Follow the exact format shown in the examples.

---
EXAMPLE 1:
Posting: We need a Product Designer (remote, US only) with 3+ years of experience in B2B SaaS. Must know Figma and have shipped at least 2 products. Salary: $120K-$140K. Reports to Head of Design. No visa sponsorship.

Output:
title: Product Designer
location: Remote (US only)
work_model: Remote
experience: 3+ years
skills: Figma, B2B SaaS, product design
salary_range: $120K-$140K
reports_to: Head of Design
visa_sponsorship: No

---
EXAMPLE 2:
Posting: Data Analyst needed for our NYC office (fully on-site). SQL and Tableau are must-haves, Python is a plus. 1-3 years experience. $85K-$100K base plus annual bonus. You'll report to the Analytics Manager. We sponsor H1B visas.

Output:
title: Data Analyst
location: NYC
work_model: On-site
experience: 1-3 years
skills: SQL, Tableau, Python (preferred)
salary_range: $85K-$100K + bonus
reports_to: Analytics Manager
visa_sponsorship: Yes

---
EXAMPLE 3:
Posting: Seeking a DevOps Engineer for our London team. Hybrid role (2 days WFH). Must have 4+ years with Kubernetes, Terraform, and CI/CD pipelines. AWS or GCP required. £75K-£90K. Reports to CTO. Open to sponsoring Skilled Worker visas.

Output:
title: DevOps Engineer
location: London
work_model: Hybrid (2 days WFH)
experience: 4+ years
skills: Kubernetes, Terraform, CI/CD, AWS or GCP
salary_range: £75K-£90K
reports_to: CTO
visa_sponsorship: Yes

---
NOW EXTRACT:
Posting: {job_posting}

Output:""")


FEW-SHOT DATA EXTRACTION (3 examples):
title: Senior Backend Engineer  
location: Austin, TX  
work_model: Hybrid (3 days in office)  
experience: 5+ years  
skills: Python or Go, PostgreSQL, Redis, AWS (ECS, Lambda, SQS)  
salary_range: $165K-$195K + equity  
reports_to: VP of Engineering  
visa_sponsorship: Yes

──────────────────────────────────────────────────
Tokens used: 596 | Est. cost: $0.000089


### Compare the two results

The few-shot version should use the **exact same field names** and **exact same format** as the examples. This consistency is what makes few-shot critical for production systems.

In [8]:
# ============================================================
# FEW-SHOT: Testing on a completely different posting
# ============================================================

new_posting = """Machine Learning Engineer — Join our AI team in San Francisco 
(hybrid, 2 days in office). We're building recommendation systems for 50M+ users. 
Need someone with 3-5 years ML experience, strong Python and PyTorch skills. 
Experience with large-scale distributed training is a big plus. Comp: $180K-$220K 
base + RSUs worth $50K/year. Reporting to Director of ML. We don't offer visa sponsorship 
at this time."""

print("FEW-SHOT on a NEW posting (not in examples):")
print("=" * 50)

result = ask(f"""Extract structured information from job postings. Follow the exact format shown in the examples.

---
EXAMPLE 1:
Posting: We need a Product Designer (remote, US only) with 3+ years of experience in B2B SaaS. Must know Figma and have shipped at least 2 products. Salary: $120K-$140K. Reports to Head of Design. No visa sponsorship.

Output:
title: Product Designer
location: Remote (US only)
work_model: Remote
experience: 3+ years
skills: Figma, B2B SaaS, product design
salary_range: $120K-$140K
reports_to: Head of Design
visa_sponsorship: No

---
EXAMPLE 2:
Posting: Data Analyst needed for our NYC office (fully on-site). SQL and Tableau are must-haves, Python is a plus. 1-3 years experience. $85K-$100K base plus annual bonus. You'll report to the Analytics Manager. We sponsor H1B visas.

Output:
title: Data Analyst
location: NYC
work_model: On-site
experience: 1-3 years
skills: SQL, Tableau, Python (preferred)
salary_range: $85K-$100K + bonus
reports_to: Analytics Manager
visa_sponsorship: Yes

---
NOW EXTRACT:
Posting: {new_posting}

Output:""")


FEW-SHOT on a NEW posting (not in examples):
title: Machine Learning Engineer  
location: San Francisco  
work_model: Hybrid (2 days in office)  
experience: 3-5 years  
skills: Python, PyTorch, machine learning, large-scale distributed training (preferred)  
salary_range: $180K-$220K + $50K/year RSUs  
reports_to: Director of ML  
visa_sponsorship: No

──────────────────────────────────────────────────
Tokens used: 463 | Est. cost: $0.000069


### Key Takeaway — Few-Shot

**When to use few-shot over zero-shot:**
- When you need a specific output format that's hard to describe in words
- When the task has subtle rules that are easier to show than explain
- When zero-shot keeps getting edge cases wrong

**How many examples?** Start with 2-3. Only add more if needed. More examples = more tokens = more cost.

**Pro tip:** Choose examples that cover **different patterns**, not similar ones. Our examples included remote, on-site, and hybrid roles across different countries and currencies.

---

# Chapter 3: Role Prompting & System Prompts

---

*Course page: Chapter 3 — Role & System Prompts (`#pe3`)*



Let's see the difference with a real scenario — **reviewing a piece of code.**

In [9]:
# ============================================================
# WITHOUT role prompting — generic response
# ============================================================

code = """def get_user_data(user_id):
    query = f"SELECT * FROM users WHERE id = {user_id}"
    result = db.execute(query)
    user = result.fetchone()
    
    orders = []
    for order in db.execute(f"SELECT * FROM orders WHERE user_id = {user_id}"):
        items = db.execute(f"SELECT * FROM items WHERE order_id = {order['id']}")
        order['items'] = items.fetchall()
        orders.append(order)
    
    user['orders'] = orders
    return user"""

print("WITHOUT ROLE — Generic response:")
print("=" * 50)
result = ask(f"Review this Python function and point out any issues:\n\n{code}")


WITHOUT ROLE — Generic response:
The provided Python function `get_user_data` has several issues and potential improvements, particularly regarding security, performance, and error handling. Here are the main points to consider:

1. **SQL Injection Vulnerability**:
   - The function constructs SQL queries using string interpolation (`f"SELECT * FROM users WHERE id = {user_id}"`). This can lead to SQL injection attacks if `user_id` is derived from user input. Instead, use parameterized queries to safely pass parameters to the SQL statements.
   ```python
   query = "SELECT * FROM users WHERE id = ?"
   result = db.execute(query, (user_id,))
   ```

2. **Error Handling**:
   - There is no error handling in place. If the user with the specified `user_id` does not exist, `result.fetchone()` will return `None`, and attempting to access `user['orders']` will raise a `TypeError`. You should check if `user` is `None` and handle this case appropriately.
   ```python
   if user is None:
       r

In [17]:
# ============================================================
# WITH role prompting — expert perspective
# ============================================================

print("WITH ROLE — Senior Backend Engineer:")
print("=" * 50)
result = ask(f"""You are a senior backend engineer with 10 years of experience building 
high-traffic web applications. You've dealt with SQL injection attacks, N+1 query 
problems that brought down production, and APIs that served 50,000 requests per second. 
You're known for code reviews that catch real production issues, not just style nits.

Review this function. For each issue you find:
1. State the problem
2. Explain the real-world consequence (what WILL go wrong in production)
3. Show the specific fix with code

```python
{code}
```""")


WITH ROLE — Senior Backend Engineer:
Here’s a review of the provided `get_user_data` function, highlighting several issues and their potential consequences in a production environment.

### Issue 1: SQL Injection Vulnerability
1. **Problem**: The function constructs SQL queries using string interpolation (`f"SELECT * FROM users WHERE id = {user_id}"`). If `user_id` is derived from user input, this opens the application to SQL injection attacks.
2. **Real-world Consequence**: An attacker could manipulate the `user_id` parameter to execute arbitrary SQL commands, potentially gaining unauthorized access to sensitive data or corrupting the database.
3. **Fix**: Use parameterized queries to safely pass user input into SQL commands. This prevents SQL injection.
   ```python
   def get_user_data(user_id):
       query = "SELECT * FROM users WHERE id = ?"
       result = db.execute(query, (user_id,))
       user = result.fetchone()
   ```

### Issue 2: N+1 Query Problem
1. **Problem**: The fun

In [18]:
# ============================================================
# SAME code, DIFFERENT roles — see how perspective changes
# ============================================================

roles = [
    {
        "title": "Security Auditor",
        "prompt": """You are an application security specialist who performs 
penetration testing and code audits. You've found vulnerabilities in Fortune 500 
apps and you think like an attacker. Your job is to find every security hole.

Audit this code for security vulnerabilities ONLY. For each finding, rate it 
as CRITICAL, HIGH, MEDIUM, or LOW and show how an attacker could exploit it."""
    },
    {
        "title": "Performance Engineer",
        "prompt": """You are a database performance engineer. Companies hire you 
when their apps slow to a crawl. You think in terms of query execution plans, 
connection pools, and response time percentiles.

Review this code for performance issues ONLY. Estimate the impact at scale 
(e.g., what happens when this serves 10,000 users with 50 orders each?)."""
    },
]

for role in roles:
    print(f"\n{'═' * 60}")
    print(f"  ROLE: {role['title']}")
    print(f"{'═' * 60}")
    result = ask(f"""{role['prompt']}

```python
{code}
```""")



════════════════════════════════════════════════════════════
  ROLE: Security Auditor
════════════════════════════════════════════════════════════
### Security Vulnerabilities Audit

1. **SQL Injection Vulnerability**
   - **Rating**: CRITICAL
   - **Description**: The code constructs SQL queries using string interpolation with user input (`user_id`). This makes the application vulnerable to SQL injection attacks. An attacker could manipulate the `user_id` parameter to execute arbitrary SQL commands.
   - **Exploitation**: For example, if an attacker provides `user_id` as `1; DROP TABLE users; --`, the query would execute the drop command, potentially compromising the database.

2. **Insecure Data Exposure**
   - **Rating**: HIGH
   - **Description**: The function retrieves all columns from the `users`, `orders`, and `items` tables without any filtering or sanitization. This could expose sensitive user information (e.g., passwords, personal details) if the database schema contains suc

In [19]:
# ============================================================
# SYSTEM PROMPT — Building a consistent assistant persona
# ============================================================

system = """You are CodeReviewBot, an AI code review assistant used by development 
teams. You review pull requests and provide actionable feedback.

Your behavior rules:
- Focus on bugs, security issues, and performance problems — not style preferences
- Always show corrected code, not just describe the problem
- Rate each issue as: 🔴 BLOCKER (must fix), 🟡 WARNING (should fix), 🔵 SUGGESTION (nice to have)
- If the code looks good, say so briefly — don't invent problems
- Keep feedback concise — developers are busy"""

code_snippets = [
    """Review this:\ndef is_admin(user):\n    if user.role == 'admin':\n        return True\n    return False""",
    """Review this:\npassword = input('Enter password: ')\nif password == 'admin123':\n    grant_access()""",
    """Review this:\ndef calculate_discount(price, discount_percent):\n    discount = price * (discount_percent / 100)\n    return round(price - discount, 2)""",
]

for snippet in code_snippets:
    print(f"\n{'═' * 60}")
    print(f"  {snippet.split(chr(10))[0]}")
    print(f"{'═' * 60}")
    result = ask(snippet, system_prompt=system)



════════════════════════════════════════════════════════════
  Review this:
════════════════════════════════════════════════════════════
The function works correctly, but it can be simplified. Here’s the corrected code:

```python
def is_admin(user):
    return user.role == 'admin'
```

This change improves readability without altering functionality. 

Rating: 🔵 SUGGESTION (nice to have)

──────────────────────────────────────────────────
Tokens: 205 (in 148, out 57) | Est. ~$0.000056

════════════════════════════════════════════════════════════
  Review this:
════════════════════════════════════════════════════════════
🔴 BLOCKER: This code has a critical security issue. Storing passwords in plain text and checking against a hardcoded value is insecure. An attacker could easily guess the password.

Here’s a corrected version that uses a hashed password for comparison:

```python
import hashlib

# Store the hashed password securely
hashed_password = hashlib.sha256('admin123'.encode()).

### Key Takeaway — Role Prompting & System Prompts

**Role prompting (in user message):** Use for one-off questions where you need a specific expert perspective.

**System prompts (separate API field):** Use for applications where the model needs consistent behavior. This is how ChatGPT, Claude, and every AI product you've used are configured.

---

# Chapter 4: Chain-of-Thought (CoT) Prompting

---

*Course page: Chapter 4 — Chain-of-Thought (`#pe4`)*


Let's use a **short word problem** where it's easy to mis-translate English into math if you rush. CoT makes the model spell out variables, the equation, and a quick sanity check.


In [ ]:
# ============================================================
# WITHOUT Chain-of-Thought — direct answer
# ============================================================

problem = """Mia is 30 years old today.

In 4 years, Mia's age will be exactly **twice** Leo's age **in 2 years** (Leo in 2 years — not in 4 years).

How old is Leo **today**?"""

print("WITHOUT CHAIN-OF-THOUGHT — Direct answer:")
print("=" * 50)
result = ask(problem)


In [ ]:
# ============================================================
# WITH Chain-of-Thought — step by step reasoning
# ============================================================

print("WITH CHAIN-OF-THOUGHT — Step by step reasoning:")
print("=" * 50)
result = ask(f"""{problem}

Work through this carefully:
1. How old will Mia be in 4 years?
2. Let L = Leo's age today. How old will Leo be in 2 years?
3. Turn the sentence into an equation ("twice as old as ...").
4. Solve for L.
5. Plug your answer back in to verify the story still makes sense.

Show each step before stating Leo's age today.""", max_tokens=800)


In [ ]:
# ============================================================
# CHAIN-OF-THOUGHT: The simple magic phrase
# ============================================================

planning_problem = """We have a team of 4 developers. We need to build: 
1) A user authentication system (estimated 3 sprints for 1 dev)
2) A payment integration (estimated 2 sprints for 1 dev, but depends on auth being done)
3) An admin dashboard (estimated 4 sprints for 1 dev, independent)
4) API documentation (estimated 1 sprint for 1 dev, depends on all features being done)
5) Testing and QA (estimated 2 sprints for 2 devs, depends on everything)

What's the minimum time to ship everything? How should we assign the team?"""

print("DIRECT ANSWER:")
print("=" * 50)
result1 = ask(planning_problem)

print("\n\nWITH 'LET'S THINK STEP BY STEP':")
print("=" * 50)
result2 = ask(planning_problem + "\n\nLet's think step by step.", max_tokens=1500)


In [ ]:
# ============================================================
# COT VARIATION: Few-shot Chain-of-Thought
# ============================================================

print("FEW-SHOT CHAIN-OF-THOUGHT:")
print("=" * 50)

result = ask(f"""Solve project planning problems step by step by identifying dependencies and the critical path.

---
EXAMPLE:
Problem: 2 developers. Tasks: Design (2 sprints, no deps), Frontend (3 sprints, needs Design), 
Backend (3 sprints, no deps), Integration (1 sprint, needs Frontend + Backend).

Reasoning:
Step 1: Map dependencies
  - Design → Frontend → Integration
  - Backend → Integration
Step 2: Find critical path (longest chain)
  - Path A: Design(2sp) + Frontend(3sp) + Integration(1sp) = 6 sprints
  - Path B: Backend(3sp) + Integration(1sp) = 4 sprints
  - Critical path: Path A = 6 sprints
Step 3: Assign team in parallel
  - Sprints 1–2: Dev1 does Design, Dev2 does Backend
  - Sprints 3–5: Dev1 does Frontend, Dev2 free (can help or do other work)
  - Sprint 6: Both devs on Integration
Answer: 6 sprints minimum. Dev2 has 2 sprints of slack.

---
NOW SOLVE:
Problem: {planning_problem}

Reasoning:""", max_tokens=1500)


### Key Takeaway — Chain-of-Thought

**When to use CoT:** Math, complex analysis, planning, multi-step reasoning, when you need to verify logic.

**When NOT to use CoT:** Simple factual questions, classification, when you need short answers.

**Three ways to trigger CoT:**
1. **Simple phrase:** "Let's think step by step"
2. **Structured steps:** "First analyze X, then consider Y, then conclude Z"
3. **Few-shot CoT:** Show an example of the reasoning process

**Trade-off:** CoT = longer responses = more tokens = higher cost. Use when quality justifies cost.

---

# Putting It All Together — Combining Techniques

---

*Stacks the techniques you just practiced. On the course page, see also **Chapter 10: Combining Techniques** (`#pe10`) for the full decision flow.*

In practice, you'll rarely use just one technique. The best prompts combine multiple techniques.


In [ ]:
# ============================================================
# COMBINED: A production-ready resume screening prompt
# ============================================================

system = """You are an automated resume screening system used by a tech company's 
recruiting team. You evaluate candidate resumes against job requirements and produce 
structured assessments.

Rules:
- Always output in the exact format shown in examples
- Rate fit as STRONG_MATCH, GOOD_MATCH, PARTIAL_MATCH, or NO_MATCH
- Be specific about gaps — never say "lacks some skills"
- Always note transferable experience even for partial matches
- Be fair and focus on skills, not pedigree"""

user_prompt = """Screen resumes against job requirements. Follow the exact format.

---
EXAMPLE:
Job: Frontend Engineer — React, TypeScript, 3+ years
Resume: 4 years at web agency. Built 15+ client sites with React and JavaScript. 
Led migration from JS to TypeScript for 3 projects. Mentored 2 junior devs.

Assessment:
FIT: STRONG_MATCH
MATCHING SKILLS: React (4 years), TypeScript (migration experience), Frontend (4 years)
GAPS: None significant
TRANSFERABLE: Team leadership, client-facing project delivery
RECOMMENDATION: Advance to technical interview

---
NOW SCREEN:
Job: GenAI Engineer — Python, LLM APIs, RAG systems, 2+ years AI/ML experience
Resume: 8 years as systems administrator. Expert in Linux, networking, and cloud 
infrastructure (AWS). Decent Python skills — wrote automation scripts and monitoring 
tools. Completed an intensive GenAI program covering LLM internals, prompt engineering, 
RAG systems, vector databases, LangChain, and deployed a capstone AI assistant project. 
No prior AI/ML work experience but strong infrastructure and production systems background.

Assessment:"""

print("COMBINED TECHNIQUES — Production Resume Screening:")
print("=" * 50)
result = ask(user_prompt, system_prompt=system)


### What we combined:

| Technique | How it was used |
|-----------|----------------|
| **System prompt** | Set the role (resume screening system) and behavioral rules |
| **Role prompting** | Defined the persona within the system prompt |
| **Few-shot** | One example showing the exact output format |
| **Output constraints** | Specific fields (FIT, MATCHING SKILLS, etc.) defined by example |

This is what production-ready prompt engineering looks like.

---

# Core Techniques — summary & quick reference

---

| Technique | What it does | When to use it | Token cost |
|-----------|-------------|----------------|------------|
| **Zero-shot** | Just describe the task | Simple, well-defined tasks | Lowest |
| **Few-shot** | Show examples of correct behavior | Format-sensitive tasks, data extraction | Medium |
| **Role / System prompt** | Set the model's persona and rules | Applications, consistent behavior | Low-Medium |
| **Chain-of-thought** | Force step-by-step reasoning | Math, troubleshooting, complex logic | Highest |

### The Prompt Engineer's Decision Flow:

```
Start with zero-shot (simplest, cheapest)
  │
  ├─ Output good enough? → Done. Ship it.
  │
  ├─ Wrong format? → Add few-shot examples
  │
  ├─ Wrong perspective/depth? → Add role prompting
  │
  ├─ Wrong reasoning? → Add chain-of-thought
  │
  └─ Still not right? → Combine techniques
```
